# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is a Croissant schema accessible at:  
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the `mlcroissant` package is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print('Dataset Name:', metadata.name)
print('Description:', metadata.description)
print('Identifier:', metadata.identifier)
print('Version:', metadata.version)
print('License:', metadata.license)


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all available record sets and their IDs
record_sets = list(dataset.record_sets)

if record_sets:
    print("Available record sets:")
    for rs in record_sets:
        print(f"- Name: {rs.name}, @id: {rs.id}")
        # List the fields/columns for each record set
        field_names = [f"{f.name} (@id: {f.id})" for f in rs.fields]
        print("  Fields:", ", ".join(field_names))
else:
    print("No record sets found in the dataset (the Croissant schema may require updating to include RecordSet definitions).")

# For demonstration, print the records for each record set
for rs in record_sets:
    print(f"\n--- Records in record set: {rs.name} (@id: {rs.id}) ---")
    for i, record in enumerate(dataset.records(record_set=rs.id)):
        if i < 3:  # only show first 3 records
            print(record)
        elif i == 3:
            print("...")
            break

## 3. Data Extraction
Load data from specific record set(s) into pandas DataFrames for analysis. All record sets and fields are referenced by their `@id`.

In [ ]:
# Extract DataFrames for record sets with tabular data
dataframes = dict()

for rs in record_sets:
    print(f"Extracting DataFrame for record set: {rs.name} (@id: {rs.id})")
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  No records found for this record set.")

# For demonstration, pick the first available tabular record set
selected_record_set_id = None
if dataframes:
    selected_record_set_id = next(iter(dataframes.keys()))  # Just pick one for EDA below
    print(f"\nSelected record set for further analysis: {selected_record_set_id}")
else:
    print("No tabular data found for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing operations—such as filtering records, normalizing numeric fields, and grouping—referencing fields by `@id`.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Run EDA only if there is a valid DataFrame
if selected_record_set_id and selected_record_set_id in dataframes:
    df = dataframes[selected_record_set_id]
    print(f"Working with DataFrame from record set ID: {selected_record_set_id}")
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

    # Attempt to auto-select a numeric field by looking for one with numeric dtype
    numeric_field = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field = col
            break
    if numeric_field is None and len(df.columns) > 0:
        # Try to coerce first column to numeric (sometimes columns are all object type)
        col = df.columns[0]
        try:
            coerced = pd.to_numeric(df[col], errors='coerce')
            if coerced.notnull().sum() > 0:
                numeric_field = col
                df[col] = coerced
        except:
            pass

    if numeric_field:
        print(f"Using numeric field: '{numeric_field}' for analysis (referenced by @id as column name)")

        # Choose a threshold as the 75th percentile for demonstration
        threshold = np.nanpercentile(df[numeric_field], 75)

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    else:
        print("No numeric fields found to analyze.")

    # Select a categorical/group field (try the first one with low number of unique values)
    group_field = None
    for col in df.select_dtypes(include=['object', 'category']).columns:
        if col != numeric_field:
            if df[col].nunique() < 10:
                group_field = col
                break
    if numeric_field and group_field is not None:
        print(f"\nGrouping filtered data by field: {group_field} (@id as column name)")
        grouped_df = (
            filtered_df.groupby(group_field)[numeric_field]
            .agg(['count', 'mean', 'std'])
            .reset_index()
        )
        print(grouped_df)
    else:
        print("No suitable group/categorical field found for grouping.")
else:
    print("No valid DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships—a histogram of the numeric field, and if available, a boxplot grouped by the chosen group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id and selected_record_set_id in dataframes and numeric_field:
    df = dataframes[selected_record_set_id]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Histogram of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field is not None and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
In this notebook, we've demonstrated how to load, explore, and analyze the FAIR² CRC survivor dataset using the `mlcroissant` library. 

- The dataset was loaded via the Croissant metadata schema, and basic metadata was inspected.
- Available record sets and their fields (referenced by their `@id`) were listed and sample records previewed.
- Tabular data was loaded into DataFrames. We performed basic EDA operations, including filtering, normalization, and grouping on fields using their `@id`s.
- Visualizations such as histograms and boxplots provided further insight into numeric and categorical field distributions.

For more in-depth analysis, you may explore domain-specific patterns, correlations, and possibly build models with the tabular data—always referencing dataset schema entities by their `@id` for reproducibility and transparency.